# agregar_valor.ipynb

Interrogatorio para integrar un activo nuevo en el sistema. Corre en Mac y edita directamente los `.py` — no se conecta a MT5, así que los datos de broker (lotaje mínimo, apalancamiento) hay que mirarlos en MT5/Windows antes de correr la celda del Paso 2.

**Alcance actual** (ajustar esta lista si se agregan scripts nuevos que necesiten mapear activos):
- `config.py`: `VALORES`, `n_sizes`, `n_sizes_ejecucion`, `A`, `B`, `PERDIDA_MAX`, `MIN_LOTAJES`, `LOTAJES_M`, `UNITS`, `APALANCAMIENTO`, `ICONOS_VALOR` (cosmético, display del monitor de X1)
- `X2_fundamentals.py`: `_YF_TICKER` (siempre), `_CRYPTO` + `_CG_ID` (solo si es crypto)

No toca `Data/`, `Data_minuto/` ni `resources/` — eso lo autocompleta X0 (backfill automático) en el primer ciclo del activo nuevo en producción.

In [ ]:
from pathlib import Path
import re

# Soporta correr el notebook desde scripts/ o desde la raíz del proyecto.
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path.cwd() / 'scripts'
CONFIG_PY = SCRIPTS_DIR / 'config.py'
X2_PY = SCRIPTS_DIR / 'X2_fundamentals.py'

# Defaults sugeridos — valores actuales uniformes entre los 6 activos ya en config.py.
# Si el "valor normal" del sistema cambia, ajustar acá antes de correr el resto del notebook.
DEFAULTS = {
    'A': 3,
    'B': 1.5,
    'PERDIDA_MAX': 200,
    'LOTAJES_M': 2,
    'n_sizes': 180,
    'n_sizes_ejecucion': 180,
    'icono': '🔹',
}


def pedir(pregunta: str, default=None, cast=str, obligatorio: bool = True):
    sugerencia = f' [{default}]' if default is not None else ''
    while True:
        resp = input(f'{pregunta}{sugerencia}: ').strip()
        if not resp:
            if default is not None:
                resp = default
            elif not obligatorio:
                return None
            else:
                print('  Respuesta obligatoria, intenta de nuevo.')
                continue
        try:
            return cast(resp)
        except (ValueError, TypeError):
            print(f'  No se pudo interpretar como {cast.__name__}, intenta de nuevo.')

## Paso 1 — identidad del activo

Si el ticker ya está en `VALORES`, se aborta sin tocar nada.

In [2]:
print('── Paso 1: identidad del activo ──')
valor = pedir('Ticker tal como lo usa el sistema (igual al símbolo en MT5, ej. BTCUSD, TSLA)').upper()

_texto_config_check = CONFIG_PY.read_text()
_valores_actuales = re.search(r"VALORES\s*=\s*\[(.*?)\]", _texto_config_check, re.S).group(1)
_valores_actuales = [v.strip().strip("'\"") for v in _valores_actuales.split(',') if v.strip()]
if valor in _valores_actuales:
    raise SystemExit(f'{valor} ya está en VALORES — abortando, no se modificó nada.')

es_crypto = pedir(f'¿{valor} es crypto? (s/n)', default='n').lower().startswith('s')
yf_ticker = pedir('Ticker en yfinance (ej. BTC-USD para crypto, TSLA para acciones)', default=valor)
cg_id = pedir('ID en CoinGecko (ej. bitcoin, ethereum)') if es_crypto else None

── Paso 1: identidad del activo ──


## Paso 2 — riesgo y trading

Lo que no tiene default sugerido hay que mirarlo en MT5/broker antes de contestar (lotaje mínimo, apalancamiento).

In [3]:
print('── Paso 2: riesgo y trading ──')
A_v = pedir('A — umbral USD para O0→OE y primer SL ganador (ver config.py)', default=DEFAULTS['A'], cast=float)
B_v = pedir('B — distancia USD del trailing stop', default=DEFAULTS['B'], cast=float)
perdida_max_v = pedir('PERDIDA_MAX — USD, se cierra la posición si la pérdida supera esto',
                      default=DEFAULTS['PERDIDA_MAX'], cast=float)
min_lotaje_v = pedir('MIN_LOTAJES — lotaje mínimo del broker para este símbolo (ver MT5)', cast=float)
lotajes_m_v = pedir('LOTAJES_M — multiplicador de lote', default=DEFAULTS['LOTAJES_M'], cast=float)
units_default = 1 if es_crypto else 100
units_v = pedir('UNITS — unidades del activo por lote', default=units_default, cast=float)
apalancamiento_v = pedir('APALANCAMIENTO — según configuración del broker para este símbolo (ver MT5)', cast=float)
n_sizes_v = pedir('n_sizes — cantidad de soportes en el grid search de X0', default=DEFAULTS['n_sizes'], cast=int)
n_sizes_ejec_v = pedir('n_sizes_ejecucion — cantidad de soportes activos en producción (X1)',
                      default=DEFAULTS['n_sizes_ejecucion'], cast=int)
icono_v = pedir('Icono/emoji para el monitor de X1', default=DEFAULTS['icono'])

── Paso 2: riesgo y trading ──


## Paso 3 — resumen y confirmación

Nada se escribe en disco hasta confirmar acá.

In [ ]:
print('── Paso 3: resumen ──')
resumen = {
    'valor': valor, 'es_crypto': es_crypto, 'yf_ticker': yf_ticker, 'cg_id': cg_id,
    'A': A_v, 'B': B_v, 'PERDIDA_MAX': perdida_max_v, 'MIN_LOTAJES': min_lotaje_v,
    'LOTAJES_M': lotajes_m_v, 'UNITS': units_v, 'APALANCAMIENTO': apalancamiento_v,
    'n_sizes': n_sizes_v, 'n_sizes_ejecucion': n_sizes_ejec_v, 'icono': icono_v,
}
for k, v in resumen.items():
    print(f'  {k}: {v}')

if not pedir('\n¿Escribir esto en config.py / X2_fundamentals.py? (s/n)',
            default='n').lower().startswith('s'):
    raise SystemExit('Cancelado — no se modificó nada.')

## Paso 4 — escritura en los scripts

`_insertar_en_literal` ignora literales dentro de bloques `""" """` (como el `n_sizes` viejo comentado en `config.py`) — sin esto, insertaría en la versión histórica en vez de la activa. `_validar_sintaxis` corta antes de escribir si la edición rompe el archivo.

In [5]:
def _lineas_activas(texto: str) -> list:
    """Máscara por línea: True si está fuera de un bloque triple-quoted. Asume que los
    delimitadores ''' o \"\"\" van solos en su propia línea, como en config.py — es un
    parser mínimo para este archivo, no uno genérico de Python."""
    activas = []
    dentro = False
    for linea in texto.splitlines():
        activas.append(not dentro)
        if linea.strip() in ('"""', "'''"):
            dentro = not dentro
    return activas


def _insertar_en_literal(texto: str, nombre: str, nueva_entrada: str) -> str:
    """Inserta `nueva_entrada` en el literal (lista/set/dict) asignado a `nombre` a nivel
    de módulo, ignorando apariciones dentro de bloques triple-quoted (configs comentadas/
    históricas). Soporta literal de una sola línea (NOMBRE = [...] / {...}) o multilínea
    (cierre en una línea con solo '}' o ']')."""
    lineas = texto.splitlines(keepends=True)
    activas = _lineas_activas(texto)
    patron = re.compile(rf'^{nombre}\s*=\s*([\[{{])')

    for i, (linea, activa) in enumerate(zip(lineas, activas)):
        if not activa:
            continue
        m = patron.match(linea)
        if not m:
            continue
        cierre = ']' if m.group(1) == '[' else '}'
        if cierre in linea[m.end():]:
            idx_cierre = linea.rindex(cierre)
            lineas[i] = linea[:idx_cierre] + nueva_entrada + linea[idx_cierre:]
            return ''.join(lineas)
        for j in range(i + 1, len(lineas)):
            if lineas[j].strip() == cierre:
                lineas.insert(j, f'    {nueva_entrada}\n')
                return ''.join(lineas)
        raise ValueError(f'No se encontró el cierre de {nombre} tras la línea {i}')
    raise ValueError(f'No se encontró {nombre} fuera de bloques comentados')


def _validar_sintaxis(texto: str, nombre_archivo: str):
    try:
        compile(texto, nombre_archivo, 'exec')
    except SyntaxError as e:
        raise SystemExit(f'Edición de {nombre_archivo} rompería la sintaxis: {e}\nNo se escribió nada.')

In [ ]:
print('── Paso 4: escribiendo en los scripts ──')

ediciones_config = [
    ('VALORES', f", '{valor}'"),
    ('n_sizes', f"'{valor}': [{n_sizes_v}],"),
    ('n_sizes_ejecucion', f"'{valor}': {n_sizes_ejec_v},"),
    ('A', f"'{valor}': {A_v},"),
    ('B', f"'{valor}': {B_v},"),
    ('PERDIDA_MAX', f"'{valor}': {perdida_max_v},"),
    ('MIN_LOTAJES', f"'{valor}': {min_lotaje_v},"),
    ('LOTAJES_M', f"'{valor}': {lotajes_m_v},"),
    ('UNITS', f"'{valor}': {units_v},"),
    ('APALANCAMIENTO', f"'{valor}': {apalancamiento_v},"),
    ('ICONOS_VALOR', f"'{valor}': '{icono_v}',"),
]
texto_config = CONFIG_PY.read_text()
for nombre, entrada in ediciones_config:
    texto_config = _insertar_en_literal(texto_config, nombre, entrada)
_validar_sintaxis(texto_config, 'config.py')
CONFIG_PY.write_text(texto_config)
print(f'config.py actualizado: {[n for n, _ in ediciones_config]}')

ediciones_x2 = [('_YF_TICKER', f"'{valor}': '{yf_ticker}',")]
if es_crypto:
    ediciones_x2.append(('_CRYPTO', f", '{valor}'"))
    ediciones_x2.append(('_CG_ID', f"'{valor}': '{cg_id}',"))
texto_x2 = X2_PY.read_text()
for nombre, entrada in ediciones_x2:
    texto_x2 = _insertar_en_literal(texto_x2, nombre, entrada)
_validar_sintaxis(texto_x2, 'X2_fundamentals.py')
X2_PY.write_text(texto_x2)
print(f'X2_fundamentals.py actualizado: {[n for n, _ in ediciones_x2]}')

print(f'\n{valor} integrado en los 2 archivos. Pendiente (manual):')
print('  1. Verificar en MT5 (Windows) que el símbolo existe y está habilitado para trading.')
print('  2. git add/commit/push desde Mac, git pull en Windows.')
print('  3. Correr X0 — el primer ciclo hace backfill automático del histórico desde FECHA_INICIAL.')